# SENA - Pelatihan awal

**Workflow:** baca fungsi dan sumber → tinjau kode/hasil tersimpan → jalankan hanya jika memang memerlukan eksperimen baru.

Sumber V3A membentuk lima kategori dari dataset, melatih IndoBERT, mengevaluasi test, menghitung confusion matrix, dan mengekspor aset. Sumber yang diterima tidak menyimpan output training. Angka SENA saat ini terdapat pada notebook 00/06; tidak dibuat sebagai output training V3A.

Kode sumber seluruh sel dipertahankan. Nama TanggapMakna pada path atau metadata historis tetap menunjukkan asal aset. File ini disusun sebagai lampiran kode; tidak wajib dijalankan ulang.


# TanggapMakna V3A — Sentence-Level Event Classifier

## Tujuan

V2 SRL cukup kuat pada:
- `FALSE-EVENT`
- `PLACE-ARG`
- `STREET-ARG`
- beberapa semantic roles lain

Tetapi local evaluation menunjukkan banyak laporan aktual menjadi `UNCERTAIN` karena **event tidak selalu disebut secara eksplisit sebagai token**.

Contoh:
> “Air naik cepat di Way Halim dan dua rumah sudah terendam.”

Kalimat seperti ini dapat menggambarkan **FLOOD** walaupun tidak memuat token `banjir`.

V3A menambahkan **sentence-level event classifier**:

`FLOOD / EARTHQUAKE / FIRE / ACCIDENT / NON_EVENT`

V3A **tidak menggantikan SRL V2**. Nantinya:

Sentence Classifier → menentukan event candidate  
SRL V2 → mengekstrak place/time/affected/victim/etc.  
Rule Engine → conflict/uncertain/completeness  
Human Verification → keputusan final

### Penting
- 60 Local Challenge Set lama **tidak digunakan untuk training**.
- V3A dibangun hanya dari dataset Mendeley 4.150 teks.
- Label sentence-level diturunkan secara deterministik dari label SRL asli.

## Referensi Teknologi yang Dipakai

- Model backbone: `indolem/indobert-base-uncased`
- Framework: Hugging Face Transformers
- Deep learning runtime: PyTorch
- Evaluasi: scikit-learn
- Data processing: pandas
- Runtime: Google Colab + GPU
- Dataset: Semantic Role Labeling Datasets for Crisis Event (Mendeley Data)

Notebook ini menyimpan konfigurasi eksperimen agar bisa dirujuk kembali pada proposal/dokumentasi.

## 1. Safe Colab setup

In [ ]:
!nvidia-smi

# Hindari upgrade paket inti Colab.
!pip install -q transformers datasets accelerate iterative-stratification scikit-learn

import os
import json
import random
import shutil
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score,
)
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("=" * 76)
print("TANGGAPMAKNA V3A — SENTENCE EVENT CLASSIFIER")
print("=" * 76)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU belum aktif. Pilih Runtime > Change runtime type > GPU."
    )

## 2. Upload dataset ZIP

Gunakan dataset yang sama:

`Semantic Role Labeling Datasets for Crisis Event.zip`

In [ ]:
from google.colab import files

uploaded = files.upload()

zip_candidates = [
    Path("/content") / name
    for name in uploaded.keys()
    if name.lower().endswith(".zip")
]

if len(zip_candidates) != 1:
    raise RuntimeError(
        "Upload tepat satu ZIP dataset."
    )

ZIP_PATH = zip_candidates[0]

print("ZIP:", ZIP_PATH)
print(
    "Size:",
    f"{ZIP_PATH.stat().st_size/(1024**2):.2f} MB"
)

## 3. Extract labeled Indonesian CSV

In [ ]:
WORK_DIR = Path("/content/tanggapmakna_v3a")
WORK_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    candidates = [
        n for n in z.namelist()
        if n.endswith("3-Labeled Data/all-labeled-data.csv")
        and "Version 2.0" not in n
    ]

    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected 1 Indonesian labeled CSV, ditemukan {len(candidates)}."
        )

    MEMBER = candidates[0]
    z.extract(MEMBER, WORK_DIR)

CSV_PATH = WORK_DIR / MEMBER

df = pd.read_csv(CSV_PATH)

print("Shape          :", df.shape)
print("Unique text_id :", df["text_id"].nunique())
print("Blank token    :", df["token"].isna().sum())

assert df["text_id"].nunique() == 4150

## 4. Cleaning minimal

In [ ]:
df_clean = (
    df[df["token"].notna()]
    .copy()
    .reset_index(drop=True)
)

print("Rows after cleaning:", len(df_clean))
print("Texts              :", df_clean["text_id"].nunique())

assert df_clean["text_id"].nunique() == 4150

## 5. Reconstruct text + derive sentence-level label

Aturan label:

- tepat satu true event dan tanpa `FALSE-EVENT`:
  - `FLOOD-EVENT` → `FLOOD`
  - `EARTHQUAKE-EVENT` → `EARTHQUAKE`
  - `FIRE-EVENT` → `FIRE`
  - `ACCIDENT-EVENT` → `ACCIDENT`

- `FALSE-EVENT` tanpa true event:
  → `NON_EVENT`

Dikeluarkan dari V3A baseline:
- teks dengan true-event + false-event bersamaan;
- teks dengan lebih dari satu true-event type;
- teks tanpa true-event maupun false-event.

Alasan: baseline classifier harus memakai target sentence-level yang tidak ambigu.

In [ ]:
EVENT_ROLE_TO_CLASS = {
    "FLOOD-EVENT": "FLOOD",
    "EARTHQUAKE-EVENT": "EARTHQUAKE",
    "FIRE-EVENT": "FIRE",
    "ACCIDENT-EVENT": "ACCIDENT",
}

TRUE_EVENT_ROLES = set(EVENT_ROLE_TO_CLASS)

records = []

for text_id, group in df_clean.groupby(
    "text_id",
    sort=False,
):
    text = " ".join(
        group["token"].astype(str).tolist()
    )

    roles = set(
        group["argument"].astype(str).tolist()
    )

    true_roles = sorted(
        roles & TRUE_EVENT_ROLES
    )

    has_false = "FALSE-EVENT" in roles

    sentence_label = None
    exclusion_reason = ""

    if (
        len(true_roles) == 1
        and not has_false
    ):
        sentence_label = EVENT_ROLE_TO_CLASS[
            true_roles[0]
        ]

    elif (
        len(true_roles) == 0
        and has_false
    ):
        sentence_label = "NON_EVENT"

    elif len(true_roles) > 1:
        exclusion_reason = "MULTIPLE_TRUE_EVENTS"

    elif true_roles and has_false:
        exclusion_reason = "TRUE_PLUS_FALSE"

    else:
        exclusion_reason = "NO_EVENT_SIGNAL"

    records.append({
        "text_id": text_id,
        "text": text,
        "sentence_label": sentence_label,
        "exclusion_reason": exclusion_reason,
        "true_event_roles": ";".join(true_roles),
        "has_false_event": has_false,
    })

records_df = pd.DataFrame(records)

print("Total original texts:", len(records_df))
print("Usable for V3A      :", records_df["sentence_label"].notna().sum())
print("Excluded             :", records_df["sentence_label"].isna().sum())

print("\nClass distribution:")
display(
    records_df[
        records_df["sentence_label"].notna()
    ]["sentence_label"]
    .value_counts()
    .to_frame("texts")
)

print("\nExclusion distribution:")
display(
    records_df[
        records_df["sentence_label"].isna()
    ]["exclusion_reason"]
    .value_counts()
    .to_frame("texts")
)

## 6. Official 3.150 / 1.000 split

Sama seperti eksperimen V2:
- `001–250` dari tiap prefix → held-out test 1.000
- sisanya → train pool 3.150

Setelah filtering sentence-label, sebagian baris excluded tetap tidak dipakai.

In [ ]:
def text_number(text_id):
    return int(
        str(text_id).split("-")[1]
    )

records_df["official_split"] = records_df[
    "text_id"
].apply(
    lambda tid: (
        "test"
        if text_number(tid) <= 250
        else "train_pool"
    )
)

usable_df = records_df[
    records_df["sentence_label"].notna()
].copy().reset_index(drop=True)

train_pool_df = usable_df[
    usable_df["official_split"] == "train_pool"
].reset_index(drop=True)

test_df = usable_df[
    usable_df["official_split"] == "test"
].reset_index(drop=True)

print("Usable train pool:", len(train_pool_df))
print("Usable test      :", len(test_df))

print("\nTRAIN POOL classes:")
display(
    train_pool_df["sentence_label"]
    .value_counts()
    .to_frame("texts")
)

print("\nTEST classes:")
display(
    test_df["sentence_label"]
    .value_counts()
    .to_frame("texts")
)

assert set(train_pool_df["text_id"]).isdisjoint(
    set(test_df["text_id"])
)

## 7. Train / validation split dari train pool

Karena ini single-label classification, kita tetap menjaga distribusi class seimbang secara proporsional.

In [ ]:
CLASS_NAMES = [
    "FLOOD",
    "EARTHQUAKE",
    "FIRE",
    "ACCIDENT",
    "NON_EVENT",
]

label2id = {
    label: i
    for i, label in enumerate(CLASS_NAMES)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

train_pool_df["label"] = train_pool_df[
    "sentence_label"
].map(label2id)

# One-hot untuk multilabel stratifier
Y = np.eye(len(CLASS_NAMES))[
    train_pool_df["label"].to_numpy()
]

X = np.arange(
    len(train_pool_df)
).reshape(-1, 1)

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED,
)

train_idx, val_idx = next(
    splitter.split(X, Y)
)

train_df = (
    train_pool_df.iloc[train_idx]
    .reset_index(drop=True)
)

val_df = (
    train_pool_df.iloc[val_idx]
    .reset_index(drop=True)
)

print("Train      :", len(train_df))
print("Validation :", len(val_df))
print("Test       :", len(test_df))

for name, frame in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    print("\n", name.upper())
    display(
        frame["sentence_label"]
        .value_counts()
        .reindex(CLASS_NAMES, fill_value=0)
        .to_frame("texts")
    )

assert set(train_df["text_id"]).isdisjoint(
    set(val_df["text_id"])
)
assert set(train_df["text_id"]).isdisjoint(
    set(test_df["text_id"])
)
assert set(val_df["text_id"]).isdisjoint(
    set(test_df["text_id"])
)

print("\n✅ No leakage.")

## 8. Load IndoBERT

Backbone:
`indolem/indobert-base-uncased`

Sequence-classification head memiliki 5 classes.

In [ ]:
MODEL_NAME = "indolem/indobert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CLASS_NAMES),
    id2label=id2label,
    label2id=label2id,
)

print("Tokenizer :", tokenizer.__class__.__name__)
print("Model     :", model.__class__.__name__)
print(
    "Parameters:",
    f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M"
)

## 9. Tokenize sequence-classification dataset

In [ ]:
def to_hf_dataset(frame):
    return Dataset.from_dict({
        "text_id": frame["text_id"].tolist(),
        "text": frame["text"].tolist(),
        "labels": frame["sentence_label"]
            .map(label2id)
            .astype(int)
            .tolist(),
    })

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


train_encoded = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "text_id",
        "text",
    ],
)

val_encoded = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "text_id",
        "text",
    ],
)

test_encoded = test_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "text_id",
        "text",
    ],
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print(train_encoded)

## 10. Check tokenized length / truncation risk

In [ ]:
lengths = []

for text in usable_df["text"]:
    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
    )
    lengths.append(
        len(encoded["input_ids"])
    )

lengths = pd.Series(lengths)

print(lengths.describe())

too_long = int(
    (lengths > MAX_LENGTH).sum()
)

print(
    f"Texts > MAX_LENGTH {MAX_LENGTH}:",
    too_long,
    "/",
    len(lengths),
)

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=40)
plt.axvline(
    MAX_LENGTH,
    linestyle="--",
)
plt.xlabel("IndoBERT token length")
plt.ylabel("Texts")
plt.title("V3A input length distribution")
plt.show()

## 11. Metrics

Fokus:
- Macro-F1 → semua class diberi bobot sama
- Accuracy
- Weighted-F1
- Per-class precision/recall/F1

Macro-F1 menjadi metric untuk memilih best checkpoint.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    pred_ids = np.argmax(
        logits,
        axis=-1,
    )

    accuracy = accuracy_score(
        labels,
        pred_ids,
    )

    p_macro, r_macro, f_macro, _ = (
        precision_recall_fscore_support(
            labels,
            pred_ids,
            labels=list(range(len(CLASS_NAMES))),
            average="macro",
            zero_division=0,
        )
    )

    _, _, f_weighted, _ = (
        precision_recall_fscore_support(
            labels,
            pred_ids,
            labels=list(range(len(CLASS_NAMES))),
            average="weighted",
            zero_division=0,
        )
    )

    return {
        "accuracy": float(accuracy),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "macro_f1": float(f_macro),
        "weighted_f1": float(f_weighted),
    }

## 12. Train V3A baseline

Baseline:
- normal Cross Entropy
- LR `2e-5`
- batch 16
- max 10 epoch
- early stopping
- weight decay 0.01
- seed 42

**Belum memakai class weights.**
Class weighting hanya dicoba jika baseline menunjukkan kelas tertentu benar-benar tertinggal.

In [ ]:
OUTPUT_DIR = "/content/tanggapmakna_v3a_sentence_classifier"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_encoded,
    eval_dataset=val_encoded,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
)

train_result = trainer.train()

print("\n" + "=" * 76)
print("V3A TRAINING SELESAI")
print("=" * 76)
print(train_result)

## 13. Evaluate held-out original TEST

Held-out test ini masih berasal dari dataset original dan **belum pernah digunakan untuk memilih checkpoint**.

In [ ]:
test_output = trainer.predict(
    test_encoded
)

test_logits = test_output.predictions
test_gold = test_output.label_ids

test_pred = np.argmax(
    test_logits,
    axis=-1,
)

test_metrics = compute_metrics(
    (
        test_logits,
        test_gold,
    )
)

print("TEST METRICS")
print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        test_gold,
        test_pred,
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

## 14. Confusion matrix

In [ ]:
cm = confusion_matrix(
    test_gold,
    test_pred,
    labels=list(range(len(CLASS_NAMES))),
)

cm_df = pd.DataFrame(
    cm,
    index=[
        f"GT_{x}"
        for x in CLASS_NAMES
    ],
    columns=[
        f"PRED_{x}"
        for x in CLASS_NAMES
    ],
)

display(cm_df)

## 15. Confidence audit

V3A nantinya tidak selalu harus auto-confirm.

Kita audit:
- confidence prediksi benar;
- confidence prediksi salah;
- coverage bila menggunakan confidence threshold.

Threshold **belum dikunci** pada cell ini.

In [ ]:
test_probs = torch.softmax(
    torch.tensor(test_logits),
    dim=-1,
).numpy()

max_conf = test_probs.max(axis=1)

audit_df = pd.DataFrame({
    "gold": [
        id2label[int(x)]
        for x in test_gold
    ],
    "pred": [
        id2label[int(x)]
        for x in test_pred
    ],
    "confidence": max_conf,
})

audit_df["correct"] = (
    audit_df["gold"]
    == audit_df["pred"]
)

print("Confidence correct predictions:")
display(
    audit_df[
        audit_df["correct"]
    ]["confidence"].describe()
)

print("\nConfidence wrong predictions:")
display(
    audit_df[
        ~audit_df["correct"]
    ]["confidence"].describe()
)

threshold_rows = []

for threshold in [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
]:
    accepted = (
        audit_df["confidence"]
        >= threshold
    )

    coverage = float(
        accepted.mean()
    )

    if accepted.sum() == 0:
        accepted_accuracy = np.nan
    else:
        accepted_accuracy = float(
            audit_df.loc[
                accepted,
                "correct",
            ].mean()
        )

    threshold_rows.append({
        "threshold": threshold,
        "coverage": coverage,
        "accepted_accuracy": accepted_accuracy,
        "accepted_count": int(accepted.sum()),
    })

threshold_df = pd.DataFrame(
    threshold_rows
)

display(threshold_df)

## 16. Handcrafted direct-report diagnostic

**Bukan final benchmark.**
Hanya untuk melihat apakah V3A mengatasi kelemahan event implisit.

Data ini tidak boleh digunakan untuk memilih hyperparameter secara agresif.

In [ ]:
diagnostic_examples = [
    "Banjir terjadi di Sukarame dan merendam beberapa rumah.",
    "Air naik cepat di Way Halim dan dua rumah sudah terendam.",
    "Jalan depan sekolah sudah tergenang dan kendaraan sulit lewat.",
    "Kebakaran terjadi di kios dekat pasar.",
    "Asap hitam dan api terlihat dari salah satu rumah di Rajabasa.",
    "Bangunan bergoyang dan warga langsung keluar rumah.",
    "Gempa terasa di Bandar Lampung sekitar pukul 14.30.",
    "Motor dan mobil bertabrakan dekat lampu merah Rajabasa.",
    "Ada motor jatuh dan pengendaranya tergeletak di jalan.",
    "Pemerintah membahas solusi banjir untuk tahun depan.",
    "Sekolah mengadakan simulasi evakuasi kebakaran.",
    "Video gempa itu ternyata rekaman lama.",
]


@torch.inference_mode()
def predict_sentence(text):
    batch = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    batch = {
        k: v.to(model.device)
        for k, v in batch.items()
    }

    logits = model(**batch).logits[0]

    probs = torch.softmax(
        logits,
        dim=-1,
    )

    pred_id = int(
        probs.argmax().item()
    )

    return {
        "pred": id2label[pred_id],
        "confidence": float(
            probs[pred_id].item()
        ),
    }


diagnostic_rows = []

for text in diagnostic_examples:
    result = predict_sentence(text)

    diagnostic_rows.append({
        "text": text,
        **result,
    })

display(
    pd.DataFrame(
        diagnostic_rows
    )
)

# ⛔ V3A CHECKPOINT

Kirim hasil berikut ke ChatGPT:

1. Cell 5 — jumlah usable/excluded + class distribution.
2. Cell 6 — class distribution train pool/test.
3. Cell 12 — output akhir training.
4. Cell 13 — TEST METRICS + classification report.
5. Cell 14 — confusion matrix.
6. Cell 15 — confidence threshold table.
7. Cell 16 — handcrafted diagnostic.

### Keputusan selanjutnya

**GO TO FUSION**
- held-out macro-F1 kuat dan relatif seimbang;
- implicit-event examples membaik;
- NON_EVENT tetap kuat.

→ gabungkan V3A classifier + V2 SRL.

**CONDITIONAL**
- beberapa class lemah;
- coba weighted CE sebagai V3A-W, hanya berdasarkan train/validation/original test analysis.

**RETHINK**
- sentence classifier gagal membedakan event/non-event secara konsisten.

## 17. Save/export V3A

In [ ]:
EXPORT_DIR = Path(
    "/content/TanggapMakna_V3A_EventClassifier"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

trainer.save_model(
    EXPORT_DIR
)

tokenizer.save_pretrained(
    EXPORT_DIR
)

metadata = {
    "project": "TanggapMakna",
    "experiment": "V3A Sentence-Level Event Classifier",
    "backbone": MODEL_NAME,
    "framework": "Hugging Face Transformers + PyTorch",
    "runtime": "Google Colab GPU",
    "classes": CLASS_NAMES,
    "label2id": label2id,
    "id2label": id2label,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "test_metrics": test_metrics,
    "training_source": "Semantic Role Labeling Datasets for Crisis Event",
    "local_challenge_used_for_training": False,
}

(EXPORT_DIR / "tanggapmakna_metadata.json").write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

archive = shutil.make_archive(
    "/content/TanggapMakna_V3A_EventClassifier",
    "zip",
    root_dir=EXPORT_DIR,
)

print("Saved:", archive)

## 18. Optional download model

In [ ]:
from google.colab import files

files.download(
    "/content/TanggapMakna_V3A_EventClassifier.zip"
)